# Risk stratification

This small jupyter notebook is used to conduct further analysze on model predictions.


In [13]:
from aidose import END_POINT_HF_DATASET_PATH
from datasets import DatasetDict, load_from_disk
import os
import pandas as pd
import numpy as np
import numpy as np
import matplotlib.pyplot as plt



from __future__ import annotations

from collections.abc import Iterable, Callable
import pandas as pd

# Load the dataset from the specified path
dataset = DatasetDict({
        "train": load_from_disk(os.path.join(END_POINT_HF_DATASET_PATH, "train")),
        "validation": load_from_disk(os.path.join(END_POINT_HF_DATASET_PATH, "validation")),
        "test": load_from_disk(os.path.join(END_POINT_HF_DATASET_PATH, "test")),
     })

In [14]:
df_predictions  = pd.read_csv("test_predictions_after_calibration.csv")

df_predictions.sample(10)

,nctid,true_label,xgb_proba,bert_proba,combined_proba,combined_pred
4578,NCT04840199,0,0.229697,0.049589,0.024766,0
3992,NCT05134649,0,0.559251,0.040238,0.073690,0
4506,NCT02859337,0,0.044794,0.001727,0.003604,0
1883,NCT04418765,0,0.741803,0.254602,0.152828,0
4223,NCT04825678,0,0.448740,0.094678,0.059200,0
281,NCT04655313,0,0.257449,0.025179,0.026415,0
4420,NCT04111536,0,0.044919,0.008062,0.003915,0
3528,NCT04760613,0,0.041479,0.169956,0.011900,0
5434,NCT04971733,0,0.107730,0.152536,0.017645,0
2830,NCT03207867,1,0.583896,0.143668,0.090982,0


In [15]:
bins = [-np.inf, 0.02, 0.05, 0.10, np.inf]
labels = [
    "lt_0_02",
    "0_02_to_0_05",
    "0_05_to_0_10",
    "ge_0_10"
]

df_predictions["proba_bin"] = pd.cut(
    df_predictions["combined_proba"],
    bins=bins,
    labels=labels,
    right=False  # [a, b)
)

# Create four separate DataFrames
df_low       = df_predictions[df_predictions["proba_bin"] == "lt_0_02"].copy()
df_medium       = df_predictions[df_predictions["proba_bin"] == "0_02_to_0_05"].copy()
df_high       = df_predictions[df_predictions["proba_bin"] == "0_05_to_0_10"].copy()
df_very_high        = df_predictions[df_predictions["proba_bin"] == "ge_0_10"].copy()

# create the test dataset as a pandas dataframe
test_set = dataset["test"].to_pandas()

In [4]:
# extract trial data for each predicted group

# low risk group
mask_low = test_set["METADATA_nctId"].isin(df_low["nctid"])
df_low_ct = test_set[mask_low].copy()

# medium risk group
mask_medium = test_set["METADATA_nctId"].isin(df_medium["nctid"])
df_medium_ct = test_set[mask_medium].copy()

# high risk group
mask_high = test_set["METADATA_nctId"].isin(df_high["nctid"])
df_high_ct = test_set[mask_high].copy()

# very high risk group
mask_very_high = test_set["METADATA_nctId"].isin(df_very_high["nctid"])
df_very_high_ct = test_set[mask_very_high].copy()


# Risk stratification

In [16]:
df_features_all = pd.concat(
    [df_low_ct, df_medium_ct, df_high_ct, df_very_high_ct],
    ignore_index=True
)

df = df_predictions.merge(
    df_features_all,
    left_on="nctid",
    right_on="METADATA_nctId",
    how="inner"
)

### Risk stratification by phases

In [17]:
PHASE_COL = "FEATURE_phases"  

def normalize_phase_list(x) -> list[int]:
    if isinstance(x, (list, tuple, set, np.ndarray, pd.Series)):
        out = []
        for v in list(x):
            if v is None or (isinstance(v, float) and np.isnan(v)):
                continue
            try:
                out.append(int(v))
            except (ValueError, TypeError):
                continue
        return out
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    try:
        return [int(x)]
    except (ValueError, TypeError):
        return []

def highest_phase(x) -> float:
    phases = normalize_phase_list(x)
    return float(max(phases)) if phases else np.nan

def map_stage(p: float) -> str:
    if pd.isna(p):
        return "Unknown"
    p = int(p)
    if p in (1, 2):
        return "Early"
    if p == 3:
        return "Mid"
    if p in (4, 5):
        return "Late"
    return "Unknown"

df["_highest_phase"] = df[PHASE_COL].apply(highest_phase)
df["stage"] = df["_highest_phase"].apply(map_stage)

df["stage"].value_counts(dropna=False)

stage
Mid        3057
Late       2332
Early       806
Unknown     123
Name: count, dtype: int64

In [18]:
def compute_stage_risk_table(
    df: pd.DataFrame,
    stage_col: str = "stage",
    bin_col: str = "proba_bin",
    label_col: str = "true_label",
    stages: tuple[str, ...] = ("Early", "Mid", "Late"),
    bin_order: tuple[str, ...] = (
        "lt_0_02",
        "0_02_to_0_05",
        "0_05_to_0_10",
        "ge_0_10"
    ),
) -> pd.DataFrame:
    """
    Compute Table-5-like risk stratification values by development stage.
    Returns a clean DataFrame ready to copy into LaTeX.
    Relative risk is computed WITHIN each stage.
    """

    # --- Clean copy ---
    d = df.copy()
    d = d[d[stage_col].isin(stages)]

    # --- Ensure correct bin ordering ---
    d[bin_col] = pd.Categorical(d[bin_col], categories=bin_order, ordered=True)

    # --- Stage prevalence ---
    stage_prev = d.groupby(stage_col)[label_col].mean().rename("stage_prevalence")

    # --- Core aggregation ---
    tab = (
        d.groupby([stage_col, bin_col])
         .agg(
             N=(label_col, "size"),
             Events=(label_col, "sum")
         )
         .reset_index()
    )

    # --- Compute metrics ---
    tab["event_rate"] = tab["Events"] / tab["N"]
    tab = tab.merge(stage_prev, on=stage_col, how="left")
    tab["relative_risk"] = tab["event_rate"] / tab["stage_prevalence"]

    # --- Formatting ---
    tab["Event rate (%)"] = (100 * tab["event_rate"]).round(2)
    tab["Relative risk"] = tab["relative_risk"].round(3)

    # --- Map risk labels ---
    label_map = {
        "lt_0_02": "Low",
        "0_02_to_0_05": "Moderate",
        "0_05_to_0_10": "High",
        "ge_0_10": "Very high"
    }

    tab["Risk group"] = tab[bin_col].map(label_map)

    # --- Final clean output ---
    final = tab[
        [
            stage_col,
            "Risk group",
            "N",
            "Events",
            "Event rate (%)",
            "Relative risk"
        ]
    ].rename(columns={
        stage_col: "Stage",
        "N": "Number of CTs",
        "Events": "Number of events"
    })

    # --- Sort properly ---
    final["Stage"] = pd.Categorical(final["Stage"], categories=stages, ordered=True)
    final["Risk group"] = pd.Categorical(
        final["Risk group"],
        categories=["Low", "Moderate", "High", "Very high"],
        ordered=True
    )

    final = final.sort_values(["Stage", "Risk group"]).reset_index(drop=True)

    return final

In [19]:
risk_stratified_by_phase = compute_stage_risk_table(df)

C:\Users\felic\AppData\Local\Temp\ipykernel_25420\1886126821.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  d[bin_col] = pd.Categorical(d[bin_col], categories=bin_order, ordered=True)
C:\Users\felic\AppData\Local\Temp\ipykernel_25420\1886126821.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  d.groupby([stage_col, bin_col])


In [9]:
risk_stratified_by_phase

,Stage,Risk group,Number of CTs,Number of events,Event rate (%),Relative risk
0,Early,Low,642,3,0.47,0.235
1,Early,Moderate,75,1,1.33,0.672
2,Early,High,50,5,10.00,5.038
3,Early,Very high,39,7,17.95,9.042
4,Mid,Low,1853,14,0.76,0.201
5,Mid,Moderate,526,15,2.85,0.758
6,Mid,High,352,32,9.09,2.417
7,Mid,Very high,326,54,16.56,4.403
8,Late,Low,934,5,0.54,0.070
9,Late,Moderate,344,10,2.91,0.379


### Risk stratification by enrollmentCount

In [10]:
def compute_enrollment_risk_table(
    df: pd.DataFrame,
    *,
    enrollment_col: str = "FEATURE_enrollmentCount",
    risk_bin_col: str = "proba_bin",
    label_col: str = "true_label",
    fixed_bins: tuple[float, ...] = (0, 50, 200, 1000, np.inf),
    fixed_labels: tuple[str, ...] = ("≤50", "51–200", "201–1000", ">1000"),
    risk_bin_order: tuple[str, ...] = ("lt_0_02", "0_02_to_0_05", "0_05_to_0_10", "ge_0_10"),
    drop_missing_enrollment: bool = True,
    within_bin_relative_risk: bool = True,
) -> pd.DataFrame:
    """
    Compute Table-5-like risk stratification values stratified by fixed enrollment bins.

    Output columns:
      Enrollment bin | Risk group | Number of CTs | Number of events | Event rate (%) | Relative risk

    Relative risk:
      - If within_bin_relative_risk=True: event_rate / prevalence within the same enrollment bin
      - Else: event_rate / overall prevalence in df

    Notes:
      - Enrollment bins are defined via pd.cut using fixed_bins and fixed_labels.
      - Risk groups are ordered using risk_bin_order, then mapped to human-readable labels.
    """
    if len(fixed_bins) != len(fixed_labels) + 1:
        raise ValueError(
            f"fixed_bins must have exactly one more element than fixed_labels "
            f"(got {len(fixed_bins)} bins and {len(fixed_labels)} labels)."
        )

    d = df.copy()

    # Enrollment cleaning
    d[enrollment_col] = pd.to_numeric(d[enrollment_col], errors="coerce")
    if drop_missing_enrollment:
        d = d.dropna(subset=[enrollment_col]).copy()

    # Ensure correct ordering of risk bins (proba_bin)
    d[risk_bin_col] = pd.Categorical(d[risk_bin_col], categories=risk_bin_order, ordered=True)

    # Fixed enrollment bins
    d["Enrollment bin"] = pd.cut(
        d[enrollment_col],
        bins=list(fixed_bins),
        labels=list(fixed_labels),
        include_lowest=True,
        right=True,
    )
    d = d.dropna(subset=["Enrollment bin"]).copy()
    d["Enrollment bin"] = pd.Categorical(d["Enrollment bin"], categories=list(fixed_labels), ordered=True)

    # Baseline prevalence(s)
    overall_prev = float(d[label_col].mean()) if len(d) else np.nan
    bin_prev = d.groupby("Enrollment bin", observed=True)[label_col].mean().rename("bin_prevalence")

    # Core aggregation
    tab = (
        d.groupby(["Enrollment bin", risk_bin_col], observed=True)
         .agg(
             **{
                 "Number of CTs": (label_col, "size"),
                 "Number of events": (label_col, "sum"),
             }
         )
         .reset_index()
    )

    tab["event_rate"] = tab["Number of events"] / tab["Number of CTs"]

    # Relative risk
    tab = tab.merge(bin_prev, on="Enrollment bin", how="left")
    denom = tab["bin_prevalence"] if within_bin_relative_risk else overall_prev
    tab["relative_risk"] = np.where(denom > 0, tab["event_rate"] / denom, np.nan)

    # Formatting
    tab["Event rate (%)"] = (100 * tab["event_rate"]).round(2)
    tab["Relative risk"] = tab["relative_risk"].round(3)

    # Human-readable risk labels
    risk_label_map = {
        "lt_0_02": "Low",
        "0_02_to_0_05": "Moderate",
        "0_05_to_0_10": "High",
        "ge_0_10": "Very high",
    }
    tab["Risk group"] = tab[risk_bin_col].map(risk_label_map)
    tab["Risk group"] = pd.Categorical(
        tab["Risk group"],
        categories=["Low", "Moderate", "High", "Very high"],
        ordered=True,
    )

    # Final selection + ordering
    final = tab[
        [
            "Enrollment bin",
            "Risk group",
            "Number of CTs",
            "Number of events",
            "Event rate (%)",
            "Relative risk",
        ]
    ].sort_values(["Enrollment bin", "Risk group"]).reset_index(drop=True)

    return final

In [12]:
enroll_table_fixed = compute_enrollment_risk_table(
    df,
    enrollment_col="FEATURE_enrollmentCount",
    fixed_bins=(0, 50, 200, 500, np.inf),
    fixed_labels=("≤50", "51–200", "201–500", ">500"),
    within_bin_relative_risk=True
)
enroll_table_fixed

,Enrollment bin,Risk group,Number of CTs,Number of events,Event rate (%),Relative risk
0,≤50,Low,2535,16,0.63,0.517
1,≤50,Moderate,328,5,1.52,1.248
2,≤50,High,124,8,6.45,5.283
3,≤50,Very high,43,8,18.60,15.236
4,51–200,Low,912,4,0.44,0.099
5,51–200,Moderate,421,13,3.09,0.698
6,51–200,High,354,32,9.04,2.042
7,51–200,Very high,301,39,12.96,2.927
8,201–500,Low,92,2,2.17,0.198
9,201–500,Moderate,149,8,5.37,0.488
